# Classification de la localisation tumorale — SVM

Ce notebook évalue un **SVM** (linéaire et RBF) pour prédire la localisation (`cort`, `dipg`, `midl`) à partir des blocs GE et CGH, avec une **nested cross-validation** pour une estimation non biaisée des performances.

## 0. Préparation

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import (
    GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
    cross_val_score, permutation_test_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
root = Path.cwd()
DATA_DIR = root.parent / "data"

SEED = 42
LABEL_ORDER = ["cort", "dipg", "midl"]

# CV externe (évaluation non biaisée)
OUTER_CV = RepeatedStratifiedKFold(n_splits=4, n_repeats=5, random_state=SEED)
# CV interne (optimisation des hyperparamètres)
INNER_CV = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)

# Grilles d'hyperparamètres
K_GE_GRID  = [40, 60, 80, 100, 120]
K_CGH_GRID = [20, 30, 40, 60]
C_GRID     = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
GAMMA_GRID = ["scale", "auto", 0.001, 0.01, 0.1]

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 1. Chargement des données

In [ ]:
def to_numeric_frame(df):
    return df.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(",", ".", regex=False), errors="coerce"))


def load_block(block_name: str, split: str) -> pd.DataFrame:
    path = DATA_DIR / f"ge_cgh_locIGR__multiblocks__{block_name}__{split}.csv"
    df = pd.read_csv(path).rename(columns={"row_id": "patient_id"}).set_index("patient_id")
    return to_numeric_frame(df)


def load_targets(split: str) -> pd.Series:
    path = DATA_DIR / f"ge_cgh_locIGR__multiblocks__y__{split}.csv"
    y_df = pd.read_csv(path).rename(columns={"row_id": "patient_id"}).set_index("patient_id")
    y = y_df[LABEL_ORDER].idxmax(axis=1)
    y.name = "localisation"
    return y


def fill_missing_from_train(train_df, test_df):
    medians = train_df.median(axis=0)
    return train_df.fillna(medians), test_df.fillna(medians)


X_ge_train  = load_block("GE", "train")
X_ge_test   = load_block("GE", "test")
X_cgh_train = load_block("CGH", "train")
X_cgh_test  = load_block("CGH", "test")
y_train     = load_targets("train")
y_test      = load_targets("test")

train_ids = X_ge_train.index.intersection(X_cgh_train.index).intersection(y_train.index)
test_ids  = X_ge_test.index.intersection(X_cgh_test.index).intersection(y_test.index)

X_ge_train  = X_ge_train.loc[train_ids]
X_ge_test   = X_ge_test.loc[test_ids]
X_cgh_train = X_cgh_train.loc[train_ids]
X_cgh_test  = X_cgh_test.loc[test_ids]
y_train     = y_train.loc[train_ids]
y_test      = y_test.loc[test_ids]

X_ge_train, X_ge_test   = fill_missing_from_train(X_ge_train, X_ge_test)
X_cgh_train, X_cgh_test = fill_missing_from_train(X_cgh_train, X_cgh_test)

print(f"Train : {len(y_train)} échantillons  |  Test : {len(y_test)} échantillons")
print(f"GE : {X_ge_train.shape[1]} features  |  CGH : {X_cgh_train.shape[1]} features")
display(y_train.value_counts().reindex(LABEL_ORDER).to_frame("train").join(
    y_test.value_counts().reindex(LABEL_ORDER).to_frame("test")))

## 2. Fonctions utilitaires

In [ ]:
def prediction_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


def bootstrap_ci(y_true, y_pred, metric_fn=balanced_accuracy_score, n_boot=2000, seed=SEED):
    rng = np.random.RandomState(seed)
    y_true_arr, y_pred_arr = np.asarray(y_true), np.asarray(y_pred)
    scores = [metric_fn(y_true_arr[idx], y_pred_arr[idx])
              for idx in (rng.choice(len(y_true_arr), len(y_true_arr), replace=True) for _ in range(n_boot))]
    lo, med, hi = np.percentile(scores, [2.5, 50, 97.5])
    return lo, med, hi


def confusion_table(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    return pd.DataFrame(matrix,
                        index=[f"true_{l}" for l in LABEL_ORDER],
                        columns=[f"pred_{l}" for l in LABEL_ORDER])


def select_features(train_df, test_df, y_train, k):
    vt = VarianceThreshold()
    train_vt = vt.fit_transform(train_df)
    test_vt  = vt.transform(test_df)
    kept = train_df.columns[vt.get_support()]

    sel = SelectKBest(score_func=f_classif, k=min(k, len(kept)))
    train_sel = sel.fit_transform(train_vt, y_train)
    test_sel  = sel.transform(test_vt)
    selected  = kept[sel.get_support()]

    return (pd.DataFrame(train_sel, index=train_df.index, columns=selected),
            pd.DataFrame(test_sel,  index=test_df.index,  columns=selected),
            selected)

## 3. Nested Cross-Validation

**Principe** : la CV interne optimise les hyperparamètres, la CV externe évalue la performance de manière non biaisée.

```
┌─ Boucle externe (évaluation) ──────────────────────┐
│  Train outer          │  Val outer (score non biaisé)│
│  ┌─ Boucle interne ─┐ │                              │
│  │ Train │ Val       │ │                              │
│  │ inner │ inner     │ │                              │
│  │ (GridSearchCV)    │ │                              │
│  └──────────────────-┘ │                              │
└────────────────────────┴──────────────────────────────┘
```

In [ ]:
def nested_cv(pipeline, param_grid, X, y, outer_cv=OUTER_CV, inner_cv=INNER_CV):
    """
    Nested cross-validation.
    Retourne les scores de la boucle externe et les meilleurs paramètres de chaque fold.
    """
    outer_scores = []
    best_params_list = []

    X_arr = np.asarray(X)

    for fold_i, (train_idx, val_idx) in enumerate(outer_cv.split(X_arr, y)):
        X_tr, X_va = X_arr[train_idx], X_arr[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        inner_search = GridSearchCV(
            pipeline, param_grid, cv=inner_cv,
            scoring="balanced_accuracy", refit=True, n_jobs=-1,
        )
        inner_search.fit(X_tr, y_tr)

        val_pred = inner_search.predict(X_va)
        score = balanced_accuracy_score(y_va, val_pred)
        outer_scores.append(score)
        best_params_list.append(inner_search.best_params_)

    return np.array(outer_scores), best_params_list


def summarize_nested_cv(name, outer_scores):
    return pd.Series({
        "model": name,
        "nested_cv_mean": np.mean(outer_scores),
        "nested_cv_std": np.std(outer_scores),
        "nested_cv_median": np.median(outer_scores),
        "nested_cv_min": np.min(outer_scores),
        "nested_cv_max": np.max(outer_scores),
    })

## 4. Pipelines SVM

Deux variantes :
- **SVM linéaire** : `kernel='linear'`, grille sur `C` et `k` (SelectKBest)
- **SVM RBF** : `kernel='rbf'`, grille sur `C`, `gamma` et nombre de composantes PCA

In [ ]:
def make_svm_linear_pipeline():
    return Pipeline([
        ("variance", VarianceThreshold()),
        ("select", SelectKBest(score_func=f_classif)),
        ("scale", StandardScaler()),
        ("clf", SVC(kernel="linear", class_weight="balanced", random_state=SEED)),
    ])


def make_svm_rbf_pipeline():
    return Pipeline([
        ("variance", VarianceThreshold()),
        ("scale", StandardScaler()),
        ("pca", PCA()),
        ("clf", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ])

## 5. Bloc GE seul

In [ ]:
# --- SVM linéaire sur GE ---
ge_lin_pipe  = make_svm_linear_pipeline()
ge_lin_grid  = {"select__k": K_GE_GRID, "clf__C": C_GRID}

print("Nested CV — SVM linéaire sur GE...")
ge_lin_scores, ge_lin_params = nested_cv(ge_lin_pipe, ge_lin_grid, X_ge_train, y_train)
print(f"  Balanced accuracy : {np.mean(ge_lin_scores):.3f} ± {np.std(ge_lin_scores):.3f}")

# --- SVM RBF sur GE ---
ge_rbf_pipe  = make_svm_rbf_pipeline()
ge_rbf_grid  = {"pca__n_components": [5, 10, 15, 20], "clf__C": C_GRID, "clf__gamma": GAMMA_GRID}

print("Nested CV — SVM RBF sur GE...")
ge_rbf_scores, ge_rbf_params = nested_cv(ge_rbf_pipe, ge_rbf_grid, X_ge_train, y_train)
print(f"  Balanced accuracy : {np.mean(ge_rbf_scores):.3f} ± {np.std(ge_rbf_scores):.3f}")

In [ ]:
# Refit du meilleur type de SVM sur tout le train, évaluation sur test
ge_best_type = "linear" if np.mean(ge_lin_scores) >= np.mean(ge_rbf_scores) else "rbf"
print(f"Meilleur SVM sur GE : {ge_best_type}")

if ge_best_type == "linear":
    ge_final = GridSearchCV(make_svm_linear_pipeline(), ge_lin_grid,
                            cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)
else:
    ge_final = GridSearchCV(make_svm_rbf_pipeline(), ge_rbf_grid,
                            cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)

ge_final.fit(X_ge_train, y_train)
ge_test_pred  = ge_final.predict(X_ge_test)
ge_train_pred = ge_final.predict(X_ge_train)

ge_test_metrics  = prediction_metrics(y_test, ge_test_pred)
ge_train_metrics = prediction_metrics(y_train, ge_train_pred)
ge_lo, ge_med, ge_hi = bootstrap_ci(y_test, ge_test_pred)

print(f"Best params : {ge_final.best_params_}")
print(f"CV interne  : {ge_final.best_score_:.3f}")
print(f"Train bal. acc. : {ge_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {ge_test_metrics['balanced_accuracy']:.3f}  CI95=[{ge_lo:.3f}, {ge_hi:.3f}]")
print()
display(confusion_table(y_test, ge_test_pred))

## 6. Bloc CGH seul

In [ ]:
# --- SVM linéaire sur CGH ---
cgh_lin_pipe = make_svm_linear_pipeline()
cgh_lin_grid = {"select__k": K_CGH_GRID, "clf__C": C_GRID}

print("Nested CV — SVM linéaire sur CGH...")
cgh_lin_scores, cgh_lin_params = nested_cv(cgh_lin_pipe, cgh_lin_grid, X_cgh_train, y_train)
print(f"  Balanced accuracy : {np.mean(cgh_lin_scores):.3f} ± {np.std(cgh_lin_scores):.3f}")

# --- SVM RBF sur CGH ---
cgh_rbf_pipe = make_svm_rbf_pipeline()
cgh_rbf_grid = {"pca__n_components": [5, 10, 15, 20], "clf__C": C_GRID, "clf__gamma": GAMMA_GRID}

print("Nested CV — SVM RBF sur CGH...")
cgh_rbf_scores, cgh_rbf_params = nested_cv(cgh_rbf_pipe, cgh_rbf_grid, X_cgh_train, y_train)
print(f"  Balanced accuracy : {np.mean(cgh_rbf_scores):.3f} ± {np.std(cgh_rbf_scores):.3f}")

In [ ]:
cgh_best_type = "linear" if np.mean(cgh_lin_scores) >= np.mean(cgh_rbf_scores) else "rbf"
print(f"Meilleur SVM sur CGH : {cgh_best_type}")

if cgh_best_type == "linear":
    cgh_final = GridSearchCV(make_svm_linear_pipeline(), cgh_lin_grid,
                             cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)
else:
    cgh_final = GridSearchCV(make_svm_rbf_pipeline(), cgh_rbf_grid,
                             cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)

cgh_final.fit(X_cgh_train, y_train)
cgh_test_pred  = cgh_final.predict(X_cgh_test)
cgh_train_pred = cgh_final.predict(X_cgh_train)

cgh_test_metrics  = prediction_metrics(y_test, cgh_test_pred)
cgh_train_metrics = prediction_metrics(y_train, cgh_train_pred)
cgh_lo, cgh_med, cgh_hi = bootstrap_ci(y_test, cgh_test_pred)

print(f"Best params : {cgh_final.best_params_}")
print(f"CV interne  : {cgh_final.best_score_:.3f}")
print(f"Train bal. acc. : {cgh_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {cgh_test_metrics['balanced_accuracy']:.3f}  CI95=[{cgh_lo:.3f}, {cgh_hi:.3f}]")
print()
display(confusion_table(y_test, cgh_test_pred))

## 7. Fusion précoce (GE + CGH concaténés)

In [ ]:
# Concaténation des deux blocs
X_all_train = pd.concat([X_ge_train.add_prefix("GE__"), X_cgh_train.add_prefix("CGH__")], axis=1)
X_all_test  = pd.concat([X_ge_test.add_prefix("GE__"),  X_cgh_test.add_prefix("CGH__")],  axis=1)

# --- SVM linéaire, fusion précoce ---
early_lin_pipe = make_svm_linear_pipeline()
early_lin_grid = {"select__k": [40, 60, 80, 100, 120, 150], "clf__C": C_GRID}

print("Nested CV — SVM linéaire, fusion précoce...")
early_lin_scores, early_lin_params = nested_cv(early_lin_pipe, early_lin_grid, X_all_train, y_train)
print(f"  Balanced accuracy : {np.mean(early_lin_scores):.3f} ± {np.std(early_lin_scores):.3f}")

# --- SVM RBF + PCA, fusion précoce ---
early_rbf_pipe = make_svm_rbf_pipeline()
early_rbf_grid = {"pca__n_components": [5, 10, 15, 20, 25], "clf__C": C_GRID, "clf__gamma": GAMMA_GRID}

print("Nested CV — SVM RBF + PCA, fusion précoce...")
early_rbf_scores, early_rbf_params = nested_cv(early_rbf_pipe, early_rbf_grid, X_all_train, y_train)
print(f"  Balanced accuracy : {np.mean(early_rbf_scores):.3f} ± {np.std(early_rbf_scores):.3f}")

In [ ]:
early_best_type = "linear" if np.mean(early_lin_scores) >= np.mean(early_rbf_scores) else "rbf"
print(f"Meilleur SVM fusion précoce : {early_best_type}")

if early_best_type == "linear":
    early_final = GridSearchCV(make_svm_linear_pipeline(), early_lin_grid,
                               cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)
else:
    early_final = GridSearchCV(make_svm_rbf_pipeline(), early_rbf_grid,
                               cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)

early_final.fit(X_all_train, y_train)
early_test_pred  = early_final.predict(X_all_test)
early_train_pred = early_final.predict(X_all_train)

early_test_metrics  = prediction_metrics(y_test, early_test_pred)
early_train_metrics = prediction_metrics(y_train, early_train_pred)
early_lo, early_med, early_hi = bootstrap_ci(y_test, early_test_pred)

print(f"Best params : {early_final.best_params_}")
print(f"CV interne  : {early_final.best_score_:.3f}")
print(f"Train bal. acc. : {early_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {early_test_metrics['balanced_accuracy']:.3f}  CI95=[{early_lo:.3f}, {early_hi:.3f}]")
print()
display(confusion_table(y_test, early_test_pred))

## 8. Fusion tardive (moyenne des décisions GE + CGH)

On combine les `decision_function` des SVM GE et CGH, pondérées par un coefficient `alpha` optimisé en CV.

In [ ]:
# Les modèles GE et CGH sont déjà entraînés (ge_final, cgh_final)
classes = ge_final.best_estimator_.named_steps["clf"].classes_

alpha_grid = np.arange(0.5, 1.01, 0.1)
best_alpha, best_alpha_score = 0.5, 0.0
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)

for alpha in alpha_grid:
    fold_scores = []
    for tr_idx, va_idx in skf.split(X_ge_train, y_train):
        d_ge  = ge_final.decision_function(X_ge_train.iloc[va_idx])
        d_cgh = cgh_final.decision_function(X_cgh_train.iloc[va_idx])
        d_blend = alpha * d_ge + (1 - alpha) * d_cgh
        preds = classes[np.argmax(d_blend, axis=1)] if d_blend.ndim > 1 else classes[(d_blend > 0).astype(int)]
        fold_scores.append(balanced_accuracy_score(y_train.iloc[va_idx], preds))
    mean_score = np.mean(fold_scores)
    if mean_score > best_alpha_score:
        best_alpha, best_alpha_score = alpha, mean_score

print(f"Meilleur alpha (poids GE) : {best_alpha:.2f}  (CV bal. acc. = {best_alpha_score:.3f})")

# Prédiction finale
d_ge_test  = ge_final.decision_function(X_ge_test)
d_cgh_test = cgh_final.decision_function(X_cgh_test)
d_late_test = best_alpha * d_ge_test + (1 - best_alpha) * d_cgh_test
late_test_pred = classes[np.argmax(d_late_test, axis=1)] if d_late_test.ndim > 1 else classes[(d_late_test > 0).astype(int)]

d_ge_train  = ge_final.decision_function(X_ge_train)
d_cgh_train = cgh_final.decision_function(X_cgh_train)
d_late_train = best_alpha * d_ge_train + (1 - best_alpha) * d_cgh_train
late_train_pred = classes[np.argmax(d_late_train, axis=1)] if d_late_train.ndim > 1 else classes[(d_late_train > 0).astype(int)]

late_test_metrics  = prediction_metrics(y_test, late_test_pred)
late_train_metrics = prediction_metrics(y_train, late_train_pred)
late_lo, late_med, late_hi = bootstrap_ci(y_test, late_test_pred)

print(f"Train bal. acc. : {late_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {late_test_metrics['balanced_accuracy']:.3f}  CI95=[{late_lo:.3f}, {late_hi:.3f}]")
print()
display(confusion_table(y_test, late_test_pred))

## 9. Tableau comparatif

In [ ]:
rows = []

def add_row(name, nested_scores, train_metrics, test_metrics, ci_lo, ci_hi):
    rows.append({
        "model": name,
        "nested_cv_bal_acc": f"{np.mean(nested_scores):.3f} ± {np.std(nested_scores):.3f}",
        "train_bal_acc": train_metrics["balanced_accuracy"],
        "test_bal_acc": test_metrics["balanced_accuracy"],
        "test_CI95": f"[{ci_lo:.3f}, {ci_hi:.3f}]",
        "test_macro_f1": test_metrics["macro_f1"],
        "train_test_gap": train_metrics["balanced_accuracy"] - test_metrics["balanced_accuracy"],
    })

# GE seul
ge_nested = ge_lin_scores if ge_best_type == "linear" else ge_rbf_scores
add_row(f"GE seul (SVM {ge_best_type})", ge_nested, ge_train_metrics, ge_test_metrics, ge_lo, ge_hi)

# CGH seul
cgh_nested = cgh_lin_scores if cgh_best_type == "linear" else cgh_rbf_scores
add_row(f"CGH seul (SVM {cgh_best_type})", cgh_nested, cgh_train_metrics, cgh_test_metrics, cgh_lo, cgh_hi)

# Fusion précoce
early_nested = early_lin_scores if early_best_type == "linear" else early_rbf_scores
add_row(f"Fusion précoce (SVM {early_best_type})", early_nested, early_train_metrics, early_test_metrics, early_lo, early_hi)

# Fusion tardive (pas de nested CV propre, on utilise le score CV de l'alpha search)
# On crée un score array artificiellement pour l'affichage
add_row("Fusion tardive (SVM)", ge_nested, late_train_metrics, late_test_metrics, late_lo, late_hi)

comparison = pd.DataFrame(rows).set_index("model")
comparison = comparison.sort_values("test_bal_acc", ascending=False)
display(comparison)

## 10. Distribution des scores nested CV

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

all_scores = {
    f"GE (SVM {ge_best_type})": ge_nested,
    f"CGH (SVM {cgh_best_type})": cgh_nested,
    f"Fusion précoce ({early_best_type})": early_nested,
}

positions = list(range(len(all_scores)))
bp = ax.boxplot(all_scores.values(), positions=positions, vert=True, patch_artist=True, widths=0.5)

colors = ["#4c72b0", "#dd8452", "#55a868"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(all_scores.keys(), rotation=15, ha="right")
ax.set_ylabel("Balanced accuracy")
ax.set_title("Scores nested CV — SVM")
ax.axhline(y=1/3, color="red", linestyle="--", alpha=0.5, label="Hasard (1/3)")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Test de permutation

On vérifie que le meilleur modèle est significativement meilleur que le hasard.

In [ ]:
# On utilise le meilleur modèle refit (pipeline complet)
best_estimator = ge_final.best_estimator_  # GE est généralement le meilleur bloc
best_X = X_ge_train

score_obs, perm_scores, pvalue = permutation_test_score(
    best_estimator, best_X, y_train,
    scoring="balanced_accuracy",
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED),
    n_permutations=500,
    random_state=SEED,
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(perm_scores, bins=30, density=True, alpha=0.7, color="steelblue", label="Permutations")
ax.axvline(score_obs, color="red", linewidth=2, label=f"Score observé = {score_obs:.3f}")
ax.set_xlabel("Balanced accuracy (CV)")
ax.set_ylabel("Densité")
ax.set_title(f"Test de permutation — SVM — p-value = {pvalue:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Score observé : {score_obs:.3f}")
print(f"p-value :       {pvalue:.4f}")

## 12. Analyse des hyperparamètres sélectionnés (nested CV)

On regarde la distribution des hyperparamètres choisis dans chaque fold externe, pour évaluer la stabilité de la sélection.

In [ ]:
# Hyperparamètres sélectionnés à chaque fold externe (GE, meilleur type)
best_params = ge_lin_params if ge_best_type == "linear" else ge_rbf_params
params_df = pd.DataFrame(best_params)

print(f"Distribution des hyperparamètres — GE (SVM {ge_best_type}) — {len(best_params)} folds externes")
for col in params_df.columns:
    print(f"\n  {col}:")
    print(f"    {params_df[col].value_counts().sort_index().to_dict()}")

## 13. Coefficients SVM linéaire (si applicable)

Pour le SVM linéaire, on peut extraire les poids des features et identifier les gènes les plus discriminants.

In [ ]:
if ge_best_type == "linear":
    best_pipe = ge_final.best_estimator_
    clf = best_pipe.named_steps["clf"]
    
    # Récupérer les noms des features sélectionnées
    vt = best_pipe.named_steps["variance"]
    sel = best_pipe.named_steps["select"]
    kept_cols = X_ge_train.columns[vt.get_support()]
    selected_cols = kept_cols[sel.get_support()]
    
    # Importance = norme L2 des coefficients par feature (sur les 3 classes)
    coef_norms = np.linalg.norm(clf.coef_, axis=0)
    importance = pd.Series(coef_norms, index=selected_cols).sort_values(ascending=False)
    
    top_n = 25
    fig, ax = plt.subplots(figsize=(8, 7))
    importance.head(top_n).plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("||coef|| (norme L2 sur les 3 classes)")
    ax.set_title(f"Top {top_n} features — SVM linéaire sur GE")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("SVM RBF sélectionné — pas de coefficients linéaires à analyser.")
    print("Les features passent par une PCA, l'interprétation directe est limitée.")